# Kernel Methods — Exercises

Companion to the note [Kernel Methods](Kernel%20Methods.md).

Practise explicit feature maps vs the kernel trick, Mercer's condition and closure rules, distances in feature space, kernel ridge regression, kernel PCA, and the scalable approximations (Nyström, random Fourier features), by hand and from scratch in NumPy.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from math import comb
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel, rbf_kernel, laplacian_kernel
from sklearn.kernel_ridge import KernelRidge
from sklearn.decomposition import KernelPCA
from sklearn.datasets import make_circles
rng = np.random.default_rng(0)
Xk = rng.normal(size=(40, 3))
Zk = rng.normal(size=(25, 3))

## Part A · Concepts

### Exercise 1 · The kernel trick in one paragraph
*🧠 Concept · ★☆☆*

Explain the kernel trick to a fellow student: what must an algorithm look like for it to apply, what does $k$ replace, and why is that a win when $\phi$ is high-dimensional?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the [[Support Vector Machines|SVM dual]]: where do the data appear?

</details>

<details>
<summary><b>✅ Solution</b></summary>

If an algorithm touches the data **only through inner products** $x_i^\top x_j$ (in training and prediction), we can replace each one by
$k(x_i,x_j)=\langle\phi(x_i),\phi(x_j)\rangle$. The algorithm then runs as if on features $\phi(x)$, giving a non-linear model in $x$, but we never build $\phi$:
computing $k$ costs $O(d)$ even when $\phi$ has millions of (or infinitely many) dimensions. The price is working with the $N\times N$ Gram matrix instead of $d$-dimensional weights.

</details>

### Exercise 2 · What the RBF kernel measures
*🧠 Concept · ★☆☆*

For $k(x,x')=\exp(-\gamma\lVert x-x'\rVert^2)$: (a) what are $k(x,x)$ and the limit for far-apart points? (b) How does $\gamma$ act as a length scale, and what
happens to a kernel model for $\gamma\to0$ and $\gamma\to\infty$? (c) Why is it called 'infinite-dimensional'?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Expand $e^{2\gamma x^\top x'}$ as a power series.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $k(x,x)=1$; $k\to0$ as $\lVert x-x'\rVert\to\infty$: it is a similarity that decays with distance.
(b) Similarity falls to $e^{-1}$ at distance $1/\sqrt\gamma$. $\gamma\to\infty$: $K\to I$, every point only resembles itself → memorisation/overfitting.
$\gamma\to0$: all points look alike (K → all ones), the model becomes nearly linear/constant → underfitting.
(c) $k=e^{-\gamma\lVert x\rVert^2}e^{-\gamma\lVert x'\rVert^2}\sum_{n\ge0}\frac{(2\gamma)^n(x^\top x')^n}{n!}$: a weighted sum of polynomial kernels of **every** degree,
so its feature map contains all monomials, infinitely many.

</details>

### Exercise 3 · Which algorithms can be kernelised?
*🧠 Concept · ★★☆*

The note lists SVMs, kernel ridge, kernel [[PCA]], [[Gaussian Processes]] and kernel k-means. State the **representer theorem** informally and use it to explain why
L2-regularised linear models can always be kernelised. Can Lasso (L1) be kernelised the same way?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The representer theorem says the minimiser of 'loss on training points + increasing function of $\lVert w\rVert$' lies in a particular subspace.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Representer theorem**: for $\min_w\sum_iL(y_i,w^\top\phi(x_i))+\Omega(\lVert w\rVert)$ with $\Omega$ increasing, an optimum has the form $w=\sum_i\alpha_i\phi(x_i)$
(any component orthogonal to the span of the data increases $\lVert w\rVert$ but not the fit). Then $w^\top\phi(x)=\sum_i\alpha_ik(x_i,x)$ and
$\lVert w\rVert^2=\alpha^\top K\alpha$: everything is in terms of $k$. L1 penalises coordinates of $w$ in the feature basis, which is not rotation invariant and not a function of
$\lVert w\rVert_2$, so the theorem does not apply and there is no direct kernel version.

</details>

### Exercise 4 · Kernels at scale
*🧠 Concept · ★★☆*

With $N=10^6$ points, how much memory does the Gram matrix need in float64? What do **Nyström** and **random Fourier features** do differently to escape the
$O(N^2)$ memory / $O(N^3)$ time? Name one weakness of each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$10^{12}$ entries × 8 bytes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$8\times10^{12}$ bytes = 8 TB, impossible. **Nyström**: pick $m\ll N$ landmark points, compute $K_{Nm}$ and $K_{mm}$ and use $K\approx K_{Nm}K_{mm}^{+}K_{mN}$ (rank $m$):
data-dependent and accurate when the spectrum decays fast, but landmark choice matters. **Random Fourier features**: for shift-invariant kernels, sample $D$ random frequencies so that
$z(x)^\top z(x')\approx k(x,x')$ (Bochner), then train a *linear* model on $z(x)$: data-independent and simple, but needs many features for a precise approximation.
Both cost $O(Nm^2)$ / $O(ND)$. See Exercises 15–16.

</details>

## Part B · By hand

### Exercise 5 · An explicit feature map
*✍️ By hand · ★☆☆*

For $x,z\in\mathbb R^2$ show that $k(x,z)=(x^\top z+1)^2=\phi(x)^\top\phi(z)$ with
$\phi(x)=(x_1^2,\ x_2^2,\ \sqrt2x_1x_2,\ \sqrt2x_1,\ \sqrt2x_2,\ 1)$. Evaluate $k$ for $x=(1,2)$, $z=(3,-1)$ both ways.

In [ ]:
k_direct = None
k_via_phi = None

_x, _z = np.array([1, 2.]), np.array([3, -1.])
check('k directly', k_direct, (_x@_z + 1)**2)
check('k via phi', k_via_phi, polynomial_kernel(_x[None], _z[None], degree=2, gamma=1, coef0=1)[0, 0])

<details>
<summary><b>💡 Hint</b></summary>

Expand $(x_1z_1+x_2z_2+1)^2$; the cross terms come with a factor 2, which the $\sqrt2$s produce.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(x_1z_1+x_2z_2+1)^2=x_1^2z_1^2+x_2^2z_2^2+2x_1x_2z_1z_2+2x_1z_1+2x_2z_2+1=\phi(x)^\top\phi(z)$. ✔
Direct: $x^\top z=3-2=1$, $k=(1+1)^2=4$. Via $\phi$: $\phi(x)=(1,4,2\sqrt2,\sqrt2,2\sqrt2,1)$, $\phi(z)=(9,1,-3\sqrt2,3\sqrt2,-\sqrt2,1)$,
dot $=9+4-12+6-4+1=4$. ✔ The kernel needs 3 multiplications, the explicit map 6-D vectors.

```python
k_direct = 4
k_via_phi = 4
```

</details>

### Exercise 6 · How big is the polynomial feature space?
*✍️ By hand · ★☆☆*

The kernel $(x^\top z+c)^p$ with $c>0$ corresponds to all monomials of degree $\le p$ in $n$ variables, of which there are $\binom{n+p}{p}$.
How many for $n=10,p=3$? For $n=100$ pixels and $p=5$? (Store both.)

In [ ]:
dim_10_3 = None
dim_100_5 = None

check('n=10, p=3', dim_10_3, comb(13, 3))
check('n=100, p=5', dim_100_5, comb(105, 5))

<details>
<summary><b>💡 Hint</b></summary>

Stars and bars: monomials of degree $\le p$ in $n$ variables ↔ degree exactly $p$ in $n+1$ variables (add a dummy variable 1).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\binom{13}{3}=286$ and $\binom{105}{5}=96\,560\,646$, about $10^8$ features, whereas the kernel costs one 100-dimensional dot product.

```python
dim_10_3 = 286
dim_100_5 = comb(105, 5)
```

</details>

### Exercise 7 · Evaluating kernels
*✍️ By hand · ★☆☆*

For $x=(0,0)$, $z=(1,2)$ and $\gamma=0.5$ compute the RBF kernel and the Laplacian kernel $\exp(-\gamma\lVert x-z\rVert_1)$, and the squared distance between
$x$ and $z$ **in RBF feature space**, $\lVert\phi(x)-\phi(z)\rVert^2=k(x,x)+k(z,z)-2k(x,z)$.

In [ ]:
k_rbf = None
k_lap = None
d2_feature = None

_x, _z = np.array([[0, 0.]]), np.array([[1, 2.]])
check('RBF', k_rbf, rbf_kernel(_x, _z, gamma=0.5)[0, 0], tol=1e-4)
check('Laplacian', k_lap, laplacian_kernel(_x, _z, gamma=0.5)[0, 0], tol=1e-4)
check('feature-space distance', d2_feature, 2 - 2*rbf_kernel(_x, _z, gamma=0.5)[0, 0], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\lVert x-z\rVert^2=5$, $\lVert x-z\rVert_1=3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

RBF: $e^{-2.5}\approx0.0821$. Laplacian: $e^{-1.5}\approx0.2231$. Feature-space distance: $1+1-2(0.0821)\approx1.836$.
For the RBF kernel all points lie on the unit sphere in feature space ($k(x,x)=1$) and squared distances are at most 2: kernel methods can compute geometry in $\phi$-space without $\phi$.

```python
k_rbf = np.exp(-2.5)
k_lap = np.exp(-1.5)
d2_feature = 2 - 2 * np.exp(-2.5)
```

</details>

### Exercise 8 · Valid kernel or not?
*✍️ By hand · ★★☆*

Mercer: $k$ is valid iff every Gram matrix is PSD. Decide (True/False):
(a) a function that gives Gram matrix $\begin{pmatrix}1&2\\2&1\end{pmatrix}$ on two points; (b) one that gives $\begin{pmatrix}2&1\\1&2\end{pmatrix}$ (could it be valid?);
(c) $k(x,z)=\lVert x-z\rVert^2$ (test $x=0$, $z=1$ in 1-D); (d) $k(x,z)=3(x^\top z)^2+e^{-\lVert x-z\rVert^2}$.

In [ ]:
valid_a = None
could_be_valid_b = None
valid_c = None
valid_d = None

check('(a)', valid_a, bool(np.linalg.eigvalsh([[1, 2], [2, 1.]]).min() >= 0))
check('(b)', could_be_valid_b, bool(np.linalg.eigvalsh([[2, 1], [1, 2.]]).min() >= 0))
check('(c)', valid_c, bool(np.linalg.eigvalsh([[0, 1], [1, 0.]]).min() >= 0))
check('(d)', valid_d, True)

<details>
<summary><b>💡 Hint</b></summary>

A symmetric $2\times2$ matrix is PSD iff its diagonal entries and its determinant are $\ge0$. For (d) use the closure rules of the note.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **False**: $\det=1-4<0$ (eigenvalues $3,-1$). (b) **Could be** (True): eigenvalues $3,1>0$; one PSD Gram matrix does not prove validity, but nothing is violated.
(c) **False**: Gram $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ has eigenvalue $-1$. Distances are not similarities. (Note $-\lVert x-z\rVert^2$ is not PSD either, but $\exp(-\gamma\lVert x-z\rVert^2)$ is.)
(d) **True**: $(x^\top z)^2$ is a product of linear kernels, scaling by $3>0$ keeps validity, and adding the RBF kernel keeps validity.

```python
valid_a = False
could_be_valid_b = True
valid_c = False
valid_d = True
```

</details>

### Exercise 9 · Closure under sums and products
*✍️ By hand · ★★☆*

Prove that if $k_1,k_2$ are valid kernels then so are (a) $ak_1$ for $a>0$, (b) $k_1+k_2$, (c) $k_1k_2$. Give the feature map in each case.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

(c) If $k_1=\langle\phi,\phi'\rangle$ and $k_2=\langle\psi,\psi'\rangle$, consider all products $\phi_i(x)\psi_j(x)$ (the tensor product).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\sqrt a\,\phi$; or: $v^\top(aK)v=a\,v^\top Kv\ge0$.
(b) Concatenate: $\phi_{+}(x)=(\phi(x),\psi(x))$ gives $\langle\phi_+(x),\phi_+(z)\rangle=k_1+k_2$; or $v^\top(K_1+K_2)v\ge0$.
(c) $\phi_\times(x)=\phi(x)\otimes\psi(x)$ with entries $\phi_i(x)\psi_j(x)$: $\sum_{ij}\phi_i(x)\psi_j(x)\phi_i(z)\psi_j(z)=k_1(x,z)k_2(x,z)$.
(Matrix view: the Hadamard product of PSD matrices is PSD, Schur product theorem.)
Consequences: polynomials with non-negative coefficients of a kernel, and $\exp(k)$ (a limit of such polynomials), are kernels, which is how the RBF kernel is shown to be valid.

</details>

### Exercise 10 · Kernel ridge regression in dual form
*✍️ By hand · ★★☆*

Ridge regression in feature space: $w=(\Phi^\top\Phi+\lambda I)^{-1}\Phi^\top y$. Show that $w=\Phi^\top\alpha$ with $\alpha=(K+\lambda I)^{-1}y$, $K=\Phi\Phi^\top$,
and hence the prediction is $f(x)=\sum_i\alpha_ik(x_i,x)$. Which matrix do you invert in each form, and when is each cheaper?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Push-through identity: $(\Phi^\top\Phi+\lambda I)\Phi^\top=\Phi^\top(\Phi\Phi^\top+\lambda I)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

From the identity, $\Phi^\top(\Phi\Phi^\top+\lambda I)^{-1}=(\Phi^\top\Phi+\lambda I)^{-1}\Phi^\top$, hence $w=\Phi^\top(K+\lambda I)^{-1}y=\Phi^\top\alpha$.
Prediction: $f(x)=w^\top\phi(x)=\alpha^\top\Phi\phi(x)=\sum_i\alpha_ik(x_i,x)$.
Primal inverts a $D\times D$ matrix (feature dimension), dual an $N\times N$ matrix: dual wins when $D>N$ (and is the only option when $D=\infty$).
This is also the [[Gaussian Processes|GP]] posterior mean with $\lambda=\sigma^2$ (see [[Linear Regression]] for the primal).

</details>

## Part C · Code from scratch

### Exercise 11 · Gram matrices
*💻 Code · ★☆☆*

Implement vectorised `k_linear(X, Z)`, `k_poly(X, Z, c, p)`, `k_rbf(X, Z, gamma)` and `k_laplace(X, Z, gamma)` returning $(n_X, n_Z)$ matrices. Compare with
`sklearn.metrics.pairwise` on `Xk, Zk`.

In [ ]:
def k_linear(X, Z):
    return None
def k_poly(X, Z, c=1.0, p=3):
    return None
def k_rbf(X, Z, gamma=0.5):
    return None
def k_laplace(X, Z, gamma=0.5):
    return None

check('linear', k_linear(Xk, Zk), linear_kernel(Xk, Zk))
check('polynomial', k_poly(Xk, Zk, 1.0, 3), polynomial_kernel(Xk, Zk, degree=3, gamma=1, coef0=1))
check('RBF', k_rbf(Xk, Zk, 0.5), rbf_kernel(Xk, Zk, gamma=0.5))
check('Laplacian', k_laplace(Xk, Zk, 0.5), laplacian_kernel(Xk, Zk, gamma=0.5))

<details>
<summary><b>💡 Hint</b></summary>

Squared distances via $\lVert x\rVert^2+\lVert z\rVert^2-2x^\top z$; L1 distances via broadcasting `np.abs(X[:, None, :] - Z[None]).sum(-1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All four reduce to one matrix product (or a broadcast) plus an element-wise function: this is why kernel methods are easy to vectorise.

```python
def _sqd(X, Z):
    return np.maximum((X ** 2).sum(1)[:, None] + (Z ** 2).sum(1)[None, :] - 2 * X @ Z.T, 0)
def k_linear(X, Z):
    return X @ Z.T
def k_poly(X, Z, c=1.0, p=3):
    return (X @ Z.T + c) ** p
def k_rbf(X, Z, gamma=0.5):
    return np.exp(-gamma * _sqd(X, Z))
def k_laplace(X, Z, gamma=0.5):
    return np.exp(-gamma * np.abs(X[:, None, :] - Z[None]).sum(-1))
```

</details>

### Exercise 12 · Checking Mercer numerically
*💻 Code · ★★☆*

Compute the smallest eigenvalue of the Gram matrix on `Xk` for (a) your RBF kernel, (b) your polynomial kernel and (c) the 'sigmoid kernel'
$\tanh(x^\top z-1)$, which is popular in older SVM code. Store the three values. Which one is not a valid kernel?

In [ ]:
min_eig_rbf = None
min_eig_poly = None
min_eig_tanh = None

check('RBF Gram is PSD', None if min_eig_rbf is None else min_eig_rbf > -1e-8, True)
check('polynomial Gram is PSD', None if min_eig_poly is None else min_eig_poly > -1e-6, True)
check('tanh Gram is indefinite', None if min_eig_tanh is None else min_eig_tanh < -1e-3, True)

<details>
<summary><b>💡 Hint</b></summary>

Use `np.linalg.eigvalsh` (symmetric matrices); tiny negative values like $-10^{-13}$ are round-off.

</details>

<details>
<summary><b>✅ Solution</b></summary>

RBF and polynomial Gram matrices have minimum eigenvalues $\ge0$ up to round-off. The tanh 'kernel' has a clearly negative eigenvalue: it is **not** a Mercer kernel
for these parameters, so the SVM dual may become non-convex. A single Gram matrix with a negative eigenvalue is enough to disprove validity; PSD on one dataset never proves it.

```python
min_eig_rbf = np.linalg.eigvalsh(k_rbf(Xk, Xk, 0.5)).min()
min_eig_poly = np.linalg.eigvalsh(k_poly(Xk, Xk, 1.0, 3)).min()
min_eig_tanh = np.linalg.eigvalsh(np.tanh(Xk @ Xk.T - 1)).min()
print(min_eig_rbf, min_eig_poly, min_eig_tanh)
```

</details>

### Exercise 13 · Kernel ridge regression
*💻 Code · ★★☆*

Implement `krr_fit(X, y, lam, gamma)` returning $\alpha=(K+\lambda I)^{-1}y$ and `krr_predict(X_train, alpha, X_new, gamma)` with the RBF kernel. Fit noisy
$\sin$ data and compare predictions on a grid with `KernelRidge(alpha=lam, kernel='rbf', gamma=gamma)`.

In [ ]:
x_kr = np.sort(rng.uniform(-3, 3, 50))[:, None]
y_kr = np.sin(2 * x_kr[:, 0]) + 0.2 * rng.normal(size=50)
x_grid = np.linspace(-3, 3, 100)[:, None]

def krr_fit(X, y, lam=0.1, gamma=1.0):
    return None

def krr_predict(Xtr, alpha, Xnew, gamma=1.0):
    return None

alpha_kr = krr_fit(x_kr, y_kr)
f_kr = None if alpha_kr is None else krr_predict(x_kr, alpha_kr, x_grid)

check('matches sklearn KernelRidge', f_kr, KernelRidge(alpha=0.1, kernel='rbf', gamma=1.0).fit(x_kr, y_kr).predict(x_grid))

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.solve(K + lam * np.eye(N), y)`; predictions `k_rbf(Xnew, Xtr) @ alpha`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A non-linear regression in 3 lines, from the derivation in Exercise 10. Larger $\lambda$ smooths the fit; larger $\gamma$ makes it wigglier. Note there is no separate intercept:
centre $y$ first if its mean is far from 0, because RBF predictions decay to 0 far from the data.

```python
x_kr = np.sort(rng.uniform(-3, 3, 50))[:, None]
y_kr = np.sin(2 * x_kr[:, 0]) + 0.2 * rng.normal(size=50)
x_grid = np.linspace(-3, 3, 100)[:, None]

def krr_fit(X, y, lam=0.1, gamma=1.0):
    return np.linalg.solve(k_rbf(X, X, gamma) + lam * np.eye(len(X)), y)

def krr_predict(Xtr, alpha, Xnew, gamma=1.0):
    return k_rbf(Xnew, Xtr, gamma) @ alpha

alpha_kr = krr_fit(x_kr, y_kr)
f_kr = krr_predict(x_kr, alpha_kr, x_grid)
```

</details>

### Exercise 14 · Kernel PCA
*💻 Code · ★★★*

Implement kernel PCA: build the RBF Gram matrix, **centre it in feature space** $\tilde K=K-\mathbf 1K-K\mathbf 1+\mathbf 1K\mathbf 1$ (with $\mathbf 1=\frac1N\mathbf{1}\mathbf{1}^\top$),
take the top-2 eigenvectors $a_k$ with eigenvalues $\lambda_k$, and return the projections $\sqrt{\lambda_k}\,a_k$. Apply it to concentric circles and compare with sklearn's `KernelPCA` (up to sign).

In [ ]:
Xc, yc = make_circles(200, factor=0.3, noise=0.05, random_state=0)

def kernel_pca(X, gamma, n_components=2):
    # TODO
    return None

Z_kpca = kernel_pca(Xc, gamma=2.0)

_ref = KernelPCA(n_components=2, kernel='rbf', gamma=2.0).fit_transform(Xc)
check('projections (up to sign)', None if Z_kpca is None else np.abs(Z_kpca), np.abs(_ref), tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

`J = np.ones((N, N)) / N`; `Kc = K - J @ K - K @ J + J @ K @ J`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.linalg.eigh` returns eigenvalues in ascending order; reverse them.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Centring in feature space is needed because PCA assumes centred data, and we cannot subtract the mean of $\phi(x)$ directly. On the circles, the first kernel principal
component already separates inner from outer ring (a linear method like [[PCA]] cannot), so a linear classifier on `Z_kpca` would work.

```python
Xc, yc = make_circles(200, factor=0.3, noise=0.05, random_state=0)

def kernel_pca(X, gamma, n_components=2):
    N = len(X)
    K = k_rbf(X, X, gamma)
    J = np.ones((N, N)) / N
    Kc = K - J @ K - K @ J + J @ K @ J
    lam, A = np.linalg.eigh(Kc)
    lam, A = lam[::-1][:n_components], A[:, ::-1][:, :n_components]
    return A * np.sqrt(lam)

Z_kpca = kernel_pca(Xc, gamma=2.0)
print('mean of PC1 by ring:', [Z_kpca[yc == k, 0].mean().round(3) for k in (0, 1)])
```

</details>

### Exercise 15 · Random Fourier features
*💻 Code · ★★☆*

For the RBF kernel with parameter $\gamma$, Bochner's theorem gives $k(x,z)\approx z(x)^\top z(z)$ with $z(x)=\sqrt{2/D}\cos(Wx+b)$, rows of $W$ drawn from
$\mathcal N(0,2\gamma I)$ and $b\sim U[0,2\pi]$. Implement `rff(X, D, gamma, seed)` and measure the max absolute error of the approximated Gram matrix on `Xk` for
$D\in\{10,100,1000,10000\}$ (same seed for all). Store the errors in `rff_err`.

In [ ]:
def rff(X, D, gamma=0.5, seed=0):
    # TODO: return the (N, D) feature matrix
    return None

D_list = [10, 100, 1000, 10000]
rff_err = None

check('error decreases with D', None if rff_err is None else bool(np.all(np.diff(rff_err) < 0)), True)
check('D=10000 is accurate (< 0.05)', None if rff_err is None else rff_err[-1] < 0.05, True)

<details>
<summary><b>💡 Hint</b></summary>

`W = r.normal(scale=np.sqrt(2 * gamma), size=(d, D))`, `b = r.uniform(0, 2*np.pi, D)`, features `np.sqrt(2/D) * np.cos(X @ W + b)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The error shrinks roughly like $1/\sqrt D$ (Monte-Carlo). A linear model (e.g. ridge) on $D$ random features costs $O(ND^2)$ instead of $O(N^3)$, and
is an unbiased approximation of kernel ridge regression.

```python
def rff(X, D, gamma=0.5, seed=0):
    r = np.random.default_rng(seed)
    W = r.normal(scale=np.sqrt(2 * gamma), size=(X.shape[1], D))
    b = r.uniform(0, 2 * np.pi, D)
    return np.sqrt(2 / D) * np.cos(X @ W + b)

D_list = [10, 100, 1000, 10000]
K_true = k_rbf(Xk, Xk, 0.5)
rff_err = []
for D in D_list:
    F = rff(Xk, D)
    rff_err.append(np.abs(F @ F.T - K_true).max())
print(np.round(rff_err, 4))
```

</details>

### Exercise 16 · Nyström approximation
*💻 Code · ★★☆*

Draw 500 points from `rng.normal(size=(500, 2))`. Implement `nystrom(X, m, gamma, seed)` returning $\hat K=C\,W^{+}C^\top$, where $m$ landmarks are sampled without replacement,
$C=K(X,\text{landmarks})$ and $W=K(\text{landmarks},\text{landmarks})$. Report the relative Frobenius error $\lVert K-\hat K\rVert_F/\lVert K\rVert_F$ for $m\in\{5,20,50,200\}$.

In [ ]:
X500 = rng.normal(size=(500, 2))
def nystrom(X, m, gamma=0.5, seed=0):
    # TODO
    return None

m_list = [5, 20, 50, 200]
nys_err = None

check('error decreases with m', None if nys_err is None else bool(np.all(np.diff(nys_err) < 0)), True)
check('m=200 is accurate (< 1%)', None if nys_err is None else nys_err[-1] < 0.01, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.pinv(W)` handles a singular $W$. Only $C$ ($N\times m$) is needed; $\hat K$ is formed here only to measure the error.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In 2-D the RBF Gram matrix has a fast-decaying spectrum (effective rank is small), so 20 landmarks already give about 3% error and 200 well under 1% (≈ 0.3%).
The error at small $m$ depends on which landmarks were drawn; k-means centres or leverage-score sampling are better than uniform sampling.

```python
X500 = rng.normal(size=(500, 2))
def nystrom(X, m, gamma=0.5, seed=0):
    r = np.random.default_rng(seed)
    L = X[r.choice(len(X), m, replace=False)]
    C = k_rbf(X, L, gamma)
    W = k_rbf(L, L, gamma)
    return C @ np.linalg.pinv(W) @ C.T

m_list = [5, 20, 50, 200]
K500 = k_rbf(X500, X500, 0.5)
nys_err = [np.linalg.norm(K500 - nystrom(X500, m)) / np.linalg.norm(K500) for m in m_list]
print(np.round(nys_err, 6))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Kernel Methods](Kernel%20Methods.md).*